<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DeepLearning/blob/main/W06_Transfer_Learning_Uebung.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Woche 6 – Transfer Learning: Von Grund auf, Feature Extraction oder Finetuning?

In diesem Notebook vergleichst du drei Strategien, um ein Bildklassifikationsmodell für eine neue Aufgabe zu trainieren:

| Strategie | Startgewichte | Was wird trainiert? |
|---|---|---|
| **Von Grund auf** (*from scratch*) | zufällig | alles |
| **Feature Extraction** | ImageNet | nur der neue Kopf (Head) |
| **Finetuning** | ImageNet | alles, Backbone mit kleinerer Lernrate |

Jede Strategie trainierst du einmal mit **100 %** und einmal mit **10 %** der Trainingsdaten.

**Anwendungsfall:** Klassifikation von Satellitenbildern in 10 Landnutzungsklassen (EuroSAT). So etwas brauchen zum Beispiel Versicherungen, Agrarbetriebe oder Raumplanungsämter.

**Aufbau:** Das Notebook folgt dem Vierschritt aus dem Projekt-Baukasten: *Fragestellung → Experiment-Design → Resultate → Interpretation*.

**Deine Aufgaben:** Du musst drei Lücken füllen (**TODO A–C**). Nach jeder Lücke folgt eine Kontrollzelle, die dir meldet, ob alles stimmt. Das Training selbst ist bereits vorbereitet.

**Vorbereitung:** *Laufzeit → Laufzeittyp ändern → T4 GPU*. Auf der GPU sollten alle Experimente zusammen nur wenige Minuten dauern, auf einer CPU rund 20 Minuten.

## 0 Setup

In [ ]:
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import EuroSAT
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms.functional import pil_to_tensor

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
print("PyTorch", torch.__version__, "| Gerät:", device)

## 1 Daten: EuroSAT

EuroSAT enthält 27'000 Satellitenbilder (Sentinel-2, RGB, 64×64 Pixel) in 10 Klassen. Quelle: Helber et al. (2019), *EuroSAT: A Novel Dataset and Deep Learning Benchmark for Land Use and Land Cover Classification*.

Damit die Experimente in der Lektion Platz haben, arbeiten wir mit einer **kleinen, stratifizierten Stichprobe**:

| Split | Bilder pro Klasse | Total |
|---|---|---|
| Training 100 % | 100 | 1'000 |
| Training 10 % | 10 (Teilmenge davon) | 100 |
| Validierung | 30 | 300 |
| Test | 100 | 1'000 |

Training, Validierung und Test überlappen sich nicht. Der 10-%-Split ist eine Teilmenge des 100-%-Splits, damit der Vergleich nur die Datenmenge verändert.

In [ ]:
dataset = EuroSAT(root="data", download=True)
CLASSES = dataset.classes
NUM_CLASSES = len(CLASSES)
print(len(dataset), "Bilder,", NUM_CLASSES, "Klassen:", CLASSES)

N_TRAIN, N_VAL, N_TEST, N_SMALL = 100, 30, 100, 10

# Stratifizierte Stichprobe: pro Klasse mischen und aufteilen
rng = np.random.default_rng(SEED)
targets = np.array(dataset.targets)
split_idx = {"train": [], "train_10": [], "val": [], "test": []}
for c in range(NUM_CLASSES):
    idx = rng.permutation(np.where(targets == c)[0])
    split_idx["train"] += list(idx[:N_TRAIN])
    split_idx["train_10"] += list(idx[:N_SMALL])                    # Teilmenge von train
    split_idx["val"] += list(idx[N_TRAIN:N_TRAIN + N_VAL])
    split_idx["test"] += list(idx[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_TEST])

def load_split(indices):
    # Bilder als uint8-Tensoren (N, 3, 64, 64) in den Speicher laden
    x = torch.stack([pil_to_tensor(dataset[i][0]) for i in indices])
    y = torch.tensor([targets[i] for i in indices])
    return x, y

data = {name: load_split(idx) for name, idx in split_idx.items()}
for name, (x, y) in data.items():
    print(f"{name:9s} {tuple(x.shape)}  Bilder pro Klasse: {torch.bincount(y).tolist()}")

In [ ]:
x, y = data["train"]
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for c, ax in enumerate(axes.flat):
    i = (y == c).nonzero()[0].item()
    ax.imshow(x[i].permute(1, 2, 0))
    ax.set_title(CLASSES[c], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2 Vorverarbeitung passend zum vortrainierten Modell

Ein vortrainiertes Modell erwartet Inputs, die so aussehen wie seine Trainingsdaten. Das sind hier ImageNet-Fotos, die mit einem bestimmten **Mittelwert und einer Standardabweichung pro Farbkanal** normalisiert wurden.

Die Gewichte bringen ihre eigene Vorverarbeitung mit. Schau sie dir an:

In [ ]:
weights = ResNet18_Weights.DEFAULT
print(weights)
print(weights.transforms())

Die Standard-Vorverarbeitung skaliert auf 224×224 Pixel. Damit die Lektion schneller läuft, vergrössern wir die 64×64-Bilder nur auf **128×128 Pixel**. Die **Normalisierung** übernehmen wir aber genau so, wie sie beim Vortraining verwendet wurde.

### TODO A: Normalisierung
Trage die Werte für `MEAN` und `STD` aus der Ausgabe oben ein (je 3 Werte, einer pro Farbkanal R, G, B).

In [ ]:
IMG_SIZE = 128

# TODO A: Mittelwert und Standardabweichung aus weights.transforms()
MEAN = None   # z. B. [..., ..., ...]
STD = None

In [ ]:
# Kontrollzelle A
if MEAN is None or STD is None:
    print("TODO A ist noch offen.")
else:
    assert np.allclose(MEAN, weights.transforms().mean) and np.allclose(STD, weights.transforms().std), \
        "Die Werte stimmen nicht mit weights.transforms() überein."
    print("TODO A ist korrekt.")

In [ ]:
def preprocess(x_uint8, normalize=True):
    # uint8 (N,3,64,64) -> float (N,3,IMG_SIZE,IMG_SIZE), normalisiert
    x = x_uint8.float() / 255.0
    x = F.interpolate(x, size=IMG_SIZE, mode="bilinear", antialias=True, align_corners=False)
    if normalize:
        mean = torch.tensor(MEAN, device=x.device).view(1, 3, 1, 1)
        std = torch.tensor(STD, device=x.device).view(1, 3, 1, 1)
        x = (x - mean) / std
    return x

def random_flip(x):
    # Leichte Data Augmentation: zufälliges horizontales und vertikales Spiegeln pro Bild
    # (bei Satellitenbildern gibt es kein «oben» und «unten»)
    n = x.shape[0]
    h = torch.rand(n, device=x.device) < 0.5
    v = torch.rand(n, device=x.device) < 0.5
    x = torch.where(h.view(-1, 1, 1, 1), x.flip(3), x)
    x = torch.where(v.view(-1, 1, 1, 1), x.flip(2), x)
    return x

## 3 Modell: Backbone und Kopf

Wir verwenden ein **ResNet-18**. Es besteht aus einem **Backbone** (Faltungsschichten, die Merkmale extrahieren) und einem **Kopf** `model.fc`, einer einzelnen linearen Schicht mit 1'000 Ausgängen für die ImageNet-Klassen.

In [ ]:
m = resnet18(weights=weights)
print("Kopf des vortrainierten Modells:", m.fc)
print("Hauptblöcke:", [name for name, _ in m.named_children()])
del m

### TODO B: Kopf ersetzen und Backbone einfrieren

Ergänze die Funktion `build_model`:

- **B1:** Bei `feature_extraction` sollen **alle** vorhandenen Parameter eingefroren werden (`requires_grad = False`).
- **B2:** Bei allen drei Strategien ersetzt du den Kopf durch eine neue lineare Schicht mit `NUM_CLASSES` Ausgängen.

*Tipp:* Die Anzahl Eingänge des alten Kopfes findest du in `model.fc.in_features`. Überlege dir, warum die Reihenfolge von B1 und B2 wichtig ist.

In [ ]:
STRATEGIES = ["scratch", "feature_extraction", "finetuning"]

def build_model(strategy):
    if strategy == "scratch":
        model = resnet18(weights=None)           # zufällige Startgewichte
    else:
        model = resnet18(weights=weights)        # ImageNet-Gewichte

    # TODO B1: bei Feature Extraction alle Parameter einfrieren
    if strategy == "feature_extraction":
        raise NotImplementedError("TODO B1")

    # TODO B2: neuen Kopf mit NUM_CLASSES Ausgängen einsetzen
    raise NotImplementedError("TODO B2")

    return model.to(device)

In [ ]:
# Kontrollzelle B
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

try:
    for s in STRATEGIES:
        mdl = build_model(s)
        total, trainable = count_params(mdl)
        print(f"{s:20s} Parameter total: {total:>10,}  trainierbar: {trainable:>10,}  ({trainable / total:.2%})")
        assert mdl.fc.out_features == NUM_CLASSES, "Der Kopf hat nicht NUM_CLASSES Ausgänge."
        if s == "feature_extraction":
            assert trainable == mdl.fc.weight.numel() + mdl.fc.bias.numel(), \
                "Bei Feature Extraction soll nur der Kopf trainierbar sein."
        else:
            assert trainable == total, "Hier sollen alle Parameter trainierbar sein."
    print("TODO B ist korrekt.")
except NotImplementedError as e:
    print(f"{e} ist noch offen.")

### TODO C: Unterschiedliche Lernraten beim Finetuning

Beim Finetuning sind die Backbone-Gewichte bereits gut. Eine zu grosse Lernrate würde sie schnell «überschreiben» (*Catastrophic Forgetting*). Der neue Kopf startet dagegen zufällig und darf schneller lernen.

Ergänze `make_optimizer` so, dass beim Finetuning **zwei Parametergruppen** entstehen: das Backbone mit `LR_BACKBONE` und der Kopf mit `LR_HEAD`.

*Tipp:* `torch.optim.AdamW` akzeptiert statt einer Parameterliste auch eine Liste von Dictionaries, z. B. `[{"params": ..., "lr": ...}, {"params": ..., "lr": ...}]`.

In [ ]:
LR_HEAD = 1e-3       # Kopf (und Training von Grund auf)
LR_BACKBONE = 1e-4   # vortrainiertes Backbone beim Finetuning: 10x kleiner

def make_optimizer(model, strategy):
    if strategy == "scratch":
        return torch.optim.AdamW(model.parameters(), lr=LR_HEAD)
    if strategy == "feature_extraction":
        # nur die trainierbaren Parameter (also den Kopf) übergeben
        return torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR_HEAD)

    # TODO C: zwei Parametergruppen für das Finetuning
    # head_params = ...
    # backbone_params = ...   (Tipp: model.named_parameters(), Namen des Kopfes beginnen mit "fc.")
    raise NotImplementedError("TODO C")

In [ ]:
# Kontrollzelle C
try:
    opt = make_optimizer(build_model("finetuning"), "finetuning")
    lrs = sorted(g["lr"] for g in opt.param_groups)
    n_params = sum(len(g["params"]) for g in opt.param_groups)
    assert len(opt.param_groups) == 2, "Es braucht genau zwei Parametergruppen."
    assert np.allclose(lrs, [LR_BACKBONE, LR_HEAD]), "Die Lernraten der Gruppen stimmen nicht."
    assert n_params == len(list(build_model("finetuning").parameters())), "Es fehlen Parameter oder sind doppelt."
    print("TODO C ist korrekt. Lernraten:", lrs)
except NotImplementedError as e:
    print(f"{e} ist noch offen.")

## 4 Training (vorbereitet)

Die Trainingsfunktion kennst du im Prinzip aus Woche 2. Neu sind zwei Details:

1. **Bestes Modell behalten:** Nach jeder Epoche wird auf dem Validierungsset gemessen. Für den Test verwenden wir die Gewichte der besten Epoche.
2. **Stolperstein BatchNorm:** `requires_grad = False` stoppt nur die Gradienten. Die BatchNorm-Schichten aktualisieren im Trainingsmodus trotzdem ihre gespeicherten Mittelwerte und Varianzen. Bei Feature Extraction wollen wir das Backbone aber unverändert lassen. Darum schalten wir die BatchNorm-Schichten dort explizit in den `eval()`-Modus.

In [ ]:
BATCH_SIZE = 32
EPOCHS = {"100%": 10, "10%": 30}   # weniger Daten -> mehr Epochen, damit das Modell genügend Updates bekommt

def freeze_batchnorm(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

@torch.no_grad()
def evaluate(model, x_uint8, y, normalize=True):
    model.eval()
    correct = 0
    for i in range(0, len(y), 256):
        xb = preprocess(x_uint8[i:i + 256].to(device), normalize)
        correct += (model(xb).argmax(1).cpu() == y[i:i + 256]).sum().item()
    return correct / len(y)

def train_model(model, strategy, train_split, epochs, optimizer=None, normalize=True, verbose=True):
    x_tr, y_tr = data[train_split]
    x_val, y_val = data["val"]
    optimizer = optimizer or make_optimizer(model, strategy)
    history, best_acc, best_state = [], -1.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        if strategy == "feature_extraction":
            freeze_batchnorm(model)
        perm = torch.randperm(len(y_tr))
        running_loss = 0.0
        for i in range(0, len(y_tr), BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            xb = random_flip(preprocess(x_tr[idx].to(device), normalize))
            yb = y_tr[idx].to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(idx)
        val_acc = evaluate(model, x_val, y_val, normalize)
        history.append({"epoch": epoch, "train_loss": running_loss / len(y_tr), "val_acc": val_acc})
        if val_acc > best_acc:
            best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
        if verbose and (epoch == 1 or epoch % 5 == 0 or epoch == epochs):
            print(f"  Epoche {epoch:2d}  Loss {history[-1]['train_loss']:.3f}  Val-Acc {val_acc:.3f}")
    model.load_state_dict(best_state)
    return history

## 5 Experiment

### Fragestellung
Wie stark verbessern vortrainierte ImageNet-Gewichte die Klassifikation von Satellitenbildern gegenüber dem Training von Grund auf? Und hängt die beste Strategie davon ab, wie viele gelabelte Trainingsbilder wir haben?

**Hypothese (vor dem Experiment notieren!):** Was erwartest du? Welche Strategie gewinnt bei 10 %, welche bei 100 %?

*Deine Hypothese:* …

### Experiment-Design
- **Variiert:** Strategie (3 Stufen) × Trainingsdatenmenge (100 % = 1'000 Bilder, 10 % = 100 Bilder), also 6 Läufe.
- **Konstant:** Architektur (ResNet-18), Splits, Vorverarbeitung, Augmentation, Batch Size, Seed, Optimizer (AdamW).
- **Metrik:** Accuracy auf dem Testset. Die Klassen sind balanciert, darum ist Accuracy hier sinnvoll. Ausgewertet wird jeweils das Modell der besten Validierungs-Epoche.
- **Einschränkung:** Pro Konfiguration gibt es nur einen Lauf (ein Seed). Wir können also keine Streuung angeben.

In [ ]:
def run(strategy, split):
    set_seed()
    model = build_model(strategy)
    t0 = time.time()
    hist = train_model(model, strategy, "train" if split == "100%" else "train_10", EPOCHS[split])
    duration = time.time() - t0
    test_acc = evaluate(model, *data["test"])
    total, trainable = count_params(model)
    return {"Strategie": strategy, "Daten": split, "Test-Acc": test_acc,
            "trainierbare Parameter": trainable, "Trainingszeit [s]": round(duration, 1),
            "history": hist}

# Kurze Prüfung, ob alle Lücken gefüllt sind
try:
    assert MEAN is not None and STD is not None
    make_optimizer(build_model("finetuning"), "finetuning")
    ready = True
except (AssertionError, NotImplementedError):
    ready = False
    print("Zuerst TODO A–C lösen, dann diese Zelle erneut ausführen.")

results = []
if ready:
    for split in ["100%", "10%"]:
        for strategy in STRATEGIES:
            print(f"--- {strategy} | {split} Trainingsdaten ---")
            results.append(run(strategy, split))

### Resultate

In [ ]:
df = pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in results])
df.pivot(index="Strategie", columns="Daten", values="Test-Acc").loc[STRATEGIES, ["10%", "100%"]].round(3)

In [ ]:
df[["Strategie", "Daten", "trainierbare Parameter", "Trainingszeit [s]"]]

In [ ]:
labels = {"scratch": "Von Grund auf", "feature_extraction": "Feature Extraction", "finetuning": "Finetuning"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, split in zip(axes, ["10%", "100%"]):
    for r in results:
        if r["Daten"] == split:
            h = pd.DataFrame(r["history"])
            ax.plot(h["epoch"], h["val_acc"], label=labels[r["Strategie"]])
    ax.set_title(f"{split} Trainingsdaten")
    ax.set_xlabel("Epoche")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Validierungs-Accuracy")
axes[0].legend()
plt.tight_layout()
plt.show()

### Interpretation

Beantworte die Fragen in 1–3 Sätzen.

**1. Welche Strategie gewinnt bei 10 % und welche bei 100 % der Daten? Wie gross ist der Abstand zum Training von Grund auf?**

*Antwort:* …

**2. Feature Extraction trainiert nur einen winzigen Teil der Parameter (siehe Tabelle). Wann lohnt sich der Mehraufwand für Finetuning?**

*Antwort:* …

**3. Wo liegt EuroSAT in der Entscheidungsmatrix (Datenmenge × Ähnlichkeit zur Quelldomäne)?**

*Antwort:* …

**4. Was erwartest du, wenn du beim Finetuning auch für das Backbone `LR = 1e-3` verwendest?** (Probiere es aus.)

*Antwort:* …

**5. Ist der Vergleich fair? Was bräuchte es für eine belastbare Aussage?**

*Antwort:* …

**6. Ein Kunde hat 200 gelabelte Drohnenbilder von Dächern und möchte Schäden erkennen. Was empfiehlst du als ersten Schritt?**

*Antwort:* …

## 6 Bonus (falls du schneller bist)

### Bonus 1: Falsche Vorverarbeitung
Was passiert bei Feature Extraction, wenn du die ImageNet-Normalisierung weglässt? Das Backbone sieht dann Werte zwischen 0 und 1 statt der gewohnten normalisierten Werte.

In [ ]:
# Tipp: train_model(...) und evaluate(...) haben ein Argument normalize=False.
# Trainiere ein Feature-Extraction-Modell auf "train_10" ohne Normalisierung und vergleiche die Test-Accuracy.

*Beobachtung:* …

### Bonus 2: Schrittweises "Auftauen" (*Gradual Unfreezing*)
Statt alles auf einmal zu trainieren, trainierst du zuerst nur den Kopf. Danach taust du schrittweise die obersten Blöcke auf (`layer4`, dann `layer3`). Die Blöcke nahe am Input bleiben eingefroren.

Implementiere drei Phasen mit je 10 Epochen auf den 10-%-Daten und vergleiche das Resultat mit der Tabelle oben.

In [ ]:
# Gerüst:
# m_gu = build_model("feature_extraction")
# for unfrozen in [[], ["layer4"], ["layer4", "layer3"]]:
#     1. requires_grad = True für die Blöcke in `unfrozen`
#     2. neuen Optimizer mit zwei Parametergruppen erstellen (nur trainierbare Parameter!)
#     3. train_model(..., optimizer=opt) für 10 Epochen auf "train_10"
# Danach Test-Accuracy ausgeben.

*Beobachtung:* …

---
**Ausblick:** Statt ImageNet-CNNs kannst du auch grössere *Foundation Models* als Backbone verwenden, zum Beispiel DINOv2 (selbstüberwachtes Vortraining) oder den Bildteil von CLIP. Viele solche Modelle findest du in der Bibliothek `timm` oder auf dem Hugging Face Hub. Lies vor dem Einsatz immer die Model Card: Auf welchen Daten wurde das Modell vortrainiert, und welche Lizenz gilt?